OPTUNA

In [ ]:
import optuna
import gc
import torch
from ultralytics import YOLO

def objective(trial):
  
    lr0 = trial.suggest_float('lr0', 1e-4, 1e-2, log=True)
    optimizer = trial.suggest_categorical('optimizer', ['AdamW', 'SGD'])
    momentum = trial.suggest_float('momentum', 0.6, 0.98)
    imgsz = trial.suggest_categorical('imgsz', [480, 544, 640])
    
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)

    
    translate = trial.suggest_float('translate', 0.0, 0.3) 
    scale = trial.suggest_float('scale', 0.0, 0.6)         
    degrees = trial.suggest_float('degrees', 0.0, 10.0)    
    mosaic = trial.suggest_float('mosaic', 0.5, 1.0)       

    
    model = YOLO('yolo11m.pt')

    def check_prune(trainer): 
        epoch = trainer.epoch
        map50 = trainer.metrics.get('metrics/mAP50(B)', 0.0)
        trial.report(map50, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
            
    model.add_callback('on_fit_epoch_end', check_prune)

    try:
        results = model.train(
            data='/kaggle/working/custom_dataset.yaml', 
            epochs=50,          
            batch=16,         
            workers=2,
            device=0,           
            save=False,         
            
           
            lr0=lr0, optimizer=optimizer, momentum=momentum, 
            imgsz=imgsz, weight_decay=weight_decay,
            
            translate=translate, scale=scale, degrees=degrees, mosaic=mosaic,
            
            
            
            fliplr=0.0, flipud=0.0, mixup=0.0, erasing=0.0, perspective=0.0,
            
            project='/kaggle/working/Optuna_YOLO11m_Aug',
            name=f'trial_{trial.number}',
            verbose=False       
        )
        
        map50 = results.box.map50
        
        
        del model
        del results
        gc.collect()                  
        torch.cuda.empty_cache()      
        
        return map50
        
    except optuna.exceptions.TrialPruned:
        del model
        gc.collect()
        torch.cuda.empty_cache()
        raise optuna.exceptions.TrialPruned()

if __name__ == "__main__":
    
    pruner = optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=3) 
    study = optuna.create_study(direction='maximize', pruner=pruner)
    study.optimize(objective, n_trials=30)
    for key, value in study.best_params.items():
        print(f"   - {key}: {value}")
    print(f"Max mAP: {study.best_value:.4f}")

TRAINING

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11m.pt')

results = model.train(
    data='/kaggle/working/custom_dataset.yaml', 
    epochs=300,         
    batch=16,
    device=0,
    

    save=True,          
    
    lr0=0.0006627165798310775,
    optimizer='SGD',
    momentum=0.9202028687937132,
    imgsz=480,
    weight_decay=0.0009324950496491481,
    
    
    translate=0.08305808905817924, scale=0.46758714565461135, degrees=0.11794494591910265, mosaic=0.8619753802601652,
    fliplr=0.0, flipud=0.0, mixup=0.0, erasing=0.0,
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, bgr=0.0,
    shear=0.0, perspective=0.0, copy_paste=0.0,
    
    project='/kaggle/working/Final_Models',
    name='Model_medium_NoAug'
)
